# Sequence Prediction - N-BACK protocol

Every stage runs with `--nback K`, which predicts a screen from **exactly `K`
preceding screens**, sliding over the trace, so the target is always the *end of
the chain*: `K=1` predicts a screen from its immediate predecessor, `K=2`
predicts screen 3 from screens 1-2, and so on.

`--nback` is an alternative to `--strict`, not a variant of it. The two flags are
mutually exclusive: `--strict` must keep reproducing Screen2Vec's own protocol
(`modeltester.py`), where a screen in the middle of the trace is predicted.

| | permissive (script default) | `--strict` | `--nback K` (this notebook) |
|---|---|---|---|
| traces kept | >= 3 screens | >= `n_predictors + 1` (= 5) | >= `K + 1` |
| context per prediction | 1..4 screens (truncated near trace start) | exactly 4 | exactly `K` |
| target | next screen | next screen, full windows only | end of the `K`-chain, full windows only |
| candidate pool | every embedding in the directory | screens of eligible traces | screens of eligible traces |
| top-k% thresholds | `ceil()` | `int()` truncation (`modeltester.py:141`) | `ceil()` |
| training negatives | any screen | screens of eligible traces | screens of eligible traces |

Because every training window is full under `--nback`, the zero-padding branch in
the loss never fires, so training and evaluation see the same context length.
That mismatch is present in the permissive pipeline.

## Both evaluations live in this notebook

This notebook produces **two** result rows from a single training run: retrieval
*without* the app-metadata half and retrieval *with* it. The cells are ordered so
that "Run All" works top to bottom and nothing is evaluated before it exists:

| step | cell | needs | produces |
|---|---|---|---|
| 1 | `9_train_joint.py` | `embeddings_4096_last/` | `EMB_DIR/`, `LSTM_PATH`, projector |
| 2 | `4_evaluate_screen_prediction.py` on `EMB_DIR` | step 1 | **result row: no app desc** |
| 3 | `6_generate_app_context_embeddings.py` | step 1 | `EMB_APP_DIR/` |
| 4 | `4_evaluate_screen_prediction.py` on `EMB_APP_DIR` | steps 1 + 3 | **result row: with app desc** |

The no-app evaluation runs before step 3 on purpose: it depends only on step 1,
while step 3 embeds every Play-Store description with SBERT and takes about an
hour. Nothing after step 2 touches `EMB_DIR`, so both rows come from the same
projector and the same LSTM checkpoint - they are a clean ablation of the app
half, not two separate runs.

### What the app half is, and why both rows are reported

`6_generate_app_context_embeddings.py` embeds each screen's Play-Store app
description with SBERT (`sentence-transformers/all-mpnet-base-v2`, 768-d) and
concatenates that vector onto the projected screen vector. It is a *per-app*
constant: every screen of the same app carries the identical second half.

The LSTM never consumes it. `9_train_joint.py` trains on the raw Qwen embeddings
only, and at evaluation time `4_evaluate_screen_prediction.py` slices the screen
part off (`ctx_raw[:, :lstm_dim]`) before the forward pass. The app half enters
*only* in the ranking step, where it is copied verbatim from the **target** onto
the prediction (`4_evaluate_screen_prediction.py:373-375`). Step 2 is that same
evaluation with the app half absent - no code change is needed, because when the
embedding dimension equals `lstm_dim` the concatenation branch does not fire.

Why it matters (see `SUMMARY.md`): with the app half attached, every candidate
from the target's app scores high, and since a RICO app has ~7.1 screens on
average (median 5), the Top-0.01% ... Top-5% columns largely measure app identity
rather than screen prediction - a measured app-only oracle already reaches 0.9968
at Top-0.1% and 1.0 at Top-1%. In the step-2 row **every** column is a
screen-level number; in the step-4 row only Top-1 and MRR are.

**Artefacts are suffixed `_nback{K}`** and cannot clobber a permissive or a
strict run (nor a run with a different `K`):

    embeddings_seq_768_nback{K}/                    (9_)
    lstm_predictor_768d_nback{K}.pth                (9_)
    sequence_projector_768d_nback{K}.pth            (9_)
    embeddings_seq_768_nback{K}_with_app_desc/      (6_)

`6_generate_app_context_embeddings.py` needs no flag of its own - it just
concatenates an app vector per screen, so it is pointed at the n-back
directories instead.

### Caveats on the numbers this produces

1. **Numbers from different `K` are not directly comparable.** The
   eligible-trace population shrinks as `K` grows (>= `K + 1` screens), and the
   candidate pool - and therefore the top-k% thresholds and the retrieval
   difficulty - shrinks with it: ~64.4k pool screens at `K=1`, ~61.7k at `K=2`,
   ~58.3k at `K=3`. A change in accuracy across `K` mixes the effect of context
   length with the effect of an easier pool. Read each `K` on its own.

2. **Screen ordering.** Screen2Vec builds each trace from an unsorted
   `os.listdir()` of `view_hierarchies`
   (`UI_embedding/precompute_embeddings.py:72`), which is not temporal order and
   is not reproducible on another filesystem. This pipeline always uses
   `gestures.json` order, i.e. true temporal order.

3. **Only the step-4 row is protocol-comparable to the paper.**
   Screen2Vec's Table 3 numbers were produced *with* the concatenated
   app-description half. Read the step-2 row against the step-4 row to size that
   effect; do not put a no-app Top-1% next to a paper Top-1% and call it a
   reproduction.

4. **Trace and layout id namespaces are joined through the index.**
   Embeddings are keyed by Rico's *global* `data/layouts/` numbering while
   `gestures.json` keys are per-trace indices; every stage here resolves them
   through `trace_layout_index.csv` (`0_build_trace_index.py`) and refuses to run
   without it. Unmatched trace screens are dropped, so the evaluated population
   is the matched subset.

In [ ]:
# ==== Configuration: set K here, everything below follows from it ====
NBACK = 2        # number of preceding screens used as context (>= 1)
PROJ_DIM = 768   # projector output dimension

EMB_DIR     = f"embeddings_seq_{PROJ_DIM}_nback{NBACK}"        # 9_ output (step 1)
EMB_APP_DIR = f"{EMB_DIR}_with_app_desc"                       # 6_ output (step 3)
LSTM_PATH   = f"lstm_predictor_{PROJ_DIM}d_nback{NBACK}.pth"   # 9_ output (step 1)

# Result-row labels. Kept inside the 45-char label column of
# print_comparison_table() so the two rows stay aligned with the paper's.
LABEL_NOAPP = f">>> Qwen3-VL+LSTM (no app) [nback={NBACK}] <<<"
LABEL_APP   = f">>> Qwen3-VL+LSTM (with app) [nback={NBACK}] <<<"

print(f"n-back = {NBACK}  |  proj_dim = {PROJ_DIM}")
print(f"  projected embeddings : {EMB_DIR}/")
print(f"  + app context        : {EMB_APP_DIR}/")
print(f"  LSTM checkpoint      : {LSTM_PATH}")

In [ ]:
# Step 1/4 - train the projector and the LSTM jointly on the raw 4096-d Qwen
# embeddings. Writes EMB_DIR/, LSTM_PATH and the projector checkpoint. Both
# evaluations below read these, so this cell must run first.
%run 9_train_joint.py --proj_dim $PROJ_DIM --nback $NBACK

In [ ]:
# Step 2/4 - evaluate WITHOUT the app-metadata half: retrieval over the plain
# PROJ_DIM-d screen vectors in EMB_DIR. Depends only on step 1, so it runs before
# the expensive SBERT stage. Every column of this row is a screen-level number.
%run 4_evaluate_screen_prediction.py --embeddings_dir $EMB_DIR --lstm_model $LSTM_PATH --nback $NBACK --label "$LABEL_NOAPP"

In [ ]:
# Step 3/4 - concatenate the 768-d SBERT app-description vector onto every screen
# vector. Reads EMB_DIR (which step 2 left untouched) and writes EMB_APP_DIR.
# Slow: it embeds every Play-Store description, roughly an hour.
%run 6_generate_app_context_embeddings.py --embeddings_dir $EMB_DIR --output_dir $EMB_APP_DIR

In [ ]:
# Step 4/4 - evaluate WITH the app-metadata half: same LSTM checkpoint as step 2,
# retrieval over the PROJ_DIM+768-d vectors from step 3. This is the
# paper-comparable row; see the caveats at the top on how to read its columns.
%run 4_evaluate_screen_prediction.py --embeddings_dir $EMB_APP_DIR --lstm_model $LSTM_PATH --nback $NBACK --label "$LABEL_APP"